# 0.0 Загрузка и предобработка данных. Функции

In [5]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from collections import defaultdict

# Текущая рабочая папка — work 17.04.2026
WORK_DIR = Path.cwd()
KIT_DIR = (WORK_DIR.parent / "eeg_segments_kit").resolve()
EXPORTS_ROOT = (WORK_DIR.parent / "exports" / "exports").resolve()
REVE_DIR = (WORK_DIR.parent / "REVE").resolve()

if str(KIT_DIR) not in sys.path:
    sys.path.insert(0, str(KIT_DIR))

import segment_loader  # из eeg_segments_kit

print("WORK_DIR     =", WORK_DIR)
print("KIT_DIR      =", KIT_DIR)
print("EXPORTS_ROOT =", EXPORTS_ROOT)
print("KIT exists   =", KIT_DIR.exists())
print("EXPORTS_ROOT exists =", EXPORTS_ROOT.exists())

# Словарь: произвольное удобное имя -> путь к папке записи (где export_manifest.json)
RECORDINGS = {
    "000":    EXPORTS_ROOT / "00000000" / "FA0183FS",
    "004":    EXPORTS_ROOT / "004" / "FA01841G",
    "005":    EXPORTS_ROOT / "005_22_03_2026" / "FA0183FS",
    "008":    EXPORTS_ROOT / "008" / "GA15011B",
    # сюда добавим позже 005_22_03_2026 / 008, когда уточним подпапки
}

for name, root in RECORDINGS.items():
    print(name, "->", root, "| exists:", root.exists())


WORK_DIR     = C:\Users\Admin\Desktop\EEG\work 22.04.2026
KIT_DIR      = C:\Users\Admin\Desktop\EEG\eeg_segments_kit
EXPORTS_ROOT = C:\Users\Admin\Desktop\EEG\exports\exports
KIT exists   = True
EXPORTS_ROOT exists = True
000 -> C:\Users\Admin\Desktop\EEG\exports\exports\00000000\FA0183FS | exists: True
004 -> C:\Users\Admin\Desktop\EEG\exports\exports\004\FA01841G | exists: True
005 -> C:\Users\Admin\Desktop\EEG\exports\exports\005_22_03_2026\FA0183FS | exists: True
008 -> C:\Users\Admin\Desktop\EEG\exports\exports\008\GA15011B | exists: True


In [2]:
def has_nonempty(intervals):
    return bool(intervals) and any(float(x["end"]) > float(x["start"]) for x in intervals)

def merge_intervals(intervals):
    """
    intervals: список dict {"start": ..., "end": ...}
    Возвращает объединённые пересекающиеся/соприкасающиеся интервалы.
    """
    if not intervals:
        return []

    arr = sorted(
        [{"start": float(x["start"]), "end": float(x["end"])} for x in intervals],
        key=lambda z: z["start"]
    )

    merged = [arr[0]]
    for cur in arr[1:]:
        prev = merged[-1]
        if cur["start"] <= prev["end"] + 1e-9:
            prev["end"] = max(prev["end"], cur["end"])
        else:
            merged.append(cur)

    return merged

def build_continuous_recording(rec_key, rec_root):
    """
    Собирает из всех сегментов записи один непрерывный массив EEG.
    Возвращает:
      data_all: np.ndarray (n_channels, n_samples_total)
      sfreq: float
      ch_names: list[str]
    """
    # Собираем информацию по сегментам
    seg_infos = []
    for idx in segment_loader.iter_segment_indices(rec_root):
        npz = segment_loader.load_segment_npz(rec_root, idx)
        t_start = float(npz["t_start_sec"])
        t_end = float(npz["t_end_sec"])
        data = npz["data"]          # (n_channels, n_times_segment)
        sfreq = float(npz["sfreq"])
        ch_names = [str(x) for x in npz["ch_names"].tolist()]
        seg_infos.append({
            "idx": idx,
            "t_start": t_start,
            "t_end": t_end,
            "data": data,
            "sfreq": sfreq,
            "ch_names": ch_names,
        })

    if not seg_infos:
        raise ValueError(f"Нет сегментов для записи {rec_key}")

    # Сортируем по глобальному времени начала
    seg_infos.sort(key=lambda s: s["t_start"])

    # Проверяем согласованность: одинаковый sfreq и ch_names
    base_sfreq = seg_infos[0]["sfreq"]
    base_ch_names = seg_infos[0]["ch_names"]
    n_channels = seg_infos[0]["data"].shape[0]

    for s in seg_infos[1:]:
        if abs(s["sfreq"] - base_sfreq) > 1e-6:
            raise ValueError(f"{rec_key}: Несовпадающая частота дискретизации между сегментами")
        if s["ch_names"] != base_ch_names:
            raise ValueError(f"{rec_key}: Несовпадающий список каналов между сегментами")
        if s["data"].shape[0] != n_channels:
            raise ValueError(f"{rec_key}: Несовпадающее число каналов")

    # Склеиваем по времени
    data_all = np.concatenate([s["data"] for s in seg_infos], axis=1)  # по оси времени

    return data_all, base_sfreq, base_ch_names

def merge_intervals(intervals, *, merge_touching=True):
    """
    intervals: список {start, end}
    Возвращает объединённые интервалы по оси времени записи.
    """
    if not intervals:
        return []

    # нормализуем и сортируем
    ivals = [
        (float(r["start"]), float(r["end"]))
        for r in intervals
        if float(r["end"]) > float(r["start"])
    ]
    ivals.sort(key=lambda x: x[0])

    merged = []
    cur_s, cur_e = ivals[0]

    for s, e in ivals[1:]:
        if merge_touching:
            # считаем [cur_s, cur_e] и [s, e] одним интервалом, если пересекаются или соприкасаются
            if s <= cur_e + 1e-9:
                cur_e = max(cur_e, e)
            else:
                merged.append({"start": cur_s, "end": cur_e})
                cur_s, cur_e = s, e
        else:
            # только если пересекаются
            if s < cur_e - 1e-9:
                cur_e = max(cur_e, e)
            else:
                merged.append({"start": cur_s, "end": cur_e})
                cur_s, cur_e = s, e

    merged.append({"start": cur_s, "end": cur_e})
    return merged


def build_intervals_for_recording_fixed(rec_key, rec_root):
    """
    Для одной записи строит:
      - интервалы сна (sleep)
      - интервалы бодрствования (wake)
      - интервалы ДЭПД (depd), объединённые по каналам
    Все интервалы в ГЛОБАЛЬНЫХ секундах от начала записи.
    """
    sleep_all = []
    wake_all = []
    depd_all = []

    for idx in segment_loader.iter_segment_indices(rec_root):
        meta = segment_loader.load_segment_meta(rec_root, idx)

        tmin_g = float(meta.get("tmin_global_sec", 0.0))
        # tmax_g = float(meta.get("tmax_global_sec", 0.0))  # при необходимости

        # Сон/бодрствование: в meta локальное время сегмента, переведём в глобальное
        for r in meta.get("intervals_sleep") or []:
            start_local = float(r["start"])
            end_local = float(r["end"])
            sleep_all.append({
                "start": tmin_g + start_local,
                "end": tmin_g + end_local,
            })

        for r in meta.get("intervals_wake") or []:
            start_local = float(r["start"])
            end_local = float(r["end"])
            wake_all.append({
                "start": tmin_g + start_local,
                "end": tmin_g + end_local,
            })

        # ДЭПД: intervals_by_channel тоже в локальном времени сегмента, вернём в глобальное
        for r in meta.get("intervals_by_channel") or []:
            start_local = float(r["start"])
            end_local = float(r["end"])
            depd_all.append({
                "start": tmin_g + start_local,
                "end": tmin_g + end_local,
                "channel": int(r["channel"]) if "channel" in r and r["channel"] is not None else None,
            })

    # Объединяем по времени (без учёта channel для депд)
    sleep_merged = merge_intervals(sleep_all)
    wake_merged = merge_intervals(wake_all)
    depd_merged = merge_intervals(
        [{"start": r["start"], "end": r["end"]} for r in depd_all]
    )

    return {
        "sleep": sleep_merged,
        "wake": wake_merged,
        "depd": depd_merged,
    }
    
def depd_span_for_recording_norm(rec_key):
    ivals = recording_intervals_norm[rec_key]["depd"]
    if not ivals:
        return None, None
    starts = [float(r["start"]) for r in ivals]
    ends = [float(r["end"]) for r in ivals]
    return min(starts), max(ends)

def clip_intervals_to_span(intervals, t_lo, t_hi):
    out = []
    for r in intervals:
        s = float(r["start"])
        e = float(r["end"])
        lo = max(s, t_lo)
        hi = min(e, t_hi)
        if hi > lo:
            out.append({"start": lo, "end": hi})
    return out

def total_duration(intervals):
    return sum(float(r["end"]) - float(r["start"]) for r in intervals)

def depd_overlap_fraction(intervals, win_start, win_end):
    """
    intervals: список dict {"start", "end"} в НОРМИРОВАННЫХ секундах записи.
    win_start, win_end: границы окна в тех же координатах.
    Возвращает долю [0,1] длины окна, занятую ДЭПД.
    """
    if not intervals:
        return 0.0
    total = 0.0
    for r in intervals:
        s = float(r["start"])
        e = float(r["end"])
        lo = max(s, win_start)
        hi = min(e, win_end)
        if hi > lo:
            total += (hi - lo)
    length = win_end - win_start
    if length <= 1e-9:
        return 0.0
    frac = total / length
    return max(0.0, min(1.0, frac))


def build_windows_for_recording_subset_trimmed(
    rec_key,
    subset,       # "sleep" или "wake"
    window_sec=0.4,
    step_sec=0.1,
):
    """
    Строит окна (0.4 с, шаг 0.1) для одной записи и одного подмножества
    (сон или бодрствование), используя:
      - сигнал из trimmed_continuous_data[rec_key]
      - интервалы из trimmed_recording_intervals[rec_key]
    Возвращает DataFrame: одна строка = одно окно.
    """
    data_all, sfreq, ch_names, span_lo, span_hi = trimmed_continuous_data[rec_key]
    ivals = trimmed_recording_intervals[rec_key]

    subset_intervals = ivals[subset]    # сон или бодрствование
    depd_intervals = ivals["depd"]      # ДЭПД (объединённые интервалы), всё в нормированных секундах

    rows = []

    for seg in subset_intervals:
        seg_start = float(seg["start"])
        seg_end = float(seg["end"])

        t = seg_start
        while t + window_sec <= seg_end + 1e-9:
            win_start = t
            win_end = t + window_sec

            start_sample = int(round(win_start * sfreq))
            end_sample = int(round(win_end * sfreq))

            if end_sample > data_all.shape[1]:
                break

            depd_frac = depd_overlap_fraction(depd_intervals, win_start, win_end)

            rows.append({
                "rec_key": rec_key,
                "subset": subset,
                "start_sec": win_start,
                "end_sec": win_end,
                "start_sample": start_sample,
                "end_sample": end_sample,
                "sfreq": float(sfreq),
                "depd_frac": depd_frac,
            })

            t += step_sec

    return pd.DataFrame(rows)

def frac_bins(df, name, bins=(0.0, 0.0, 0.1, 0.5, 1.0)):
    # bins: точки раздела, интерпретируем как [0,0], (0,0.1], (0.1,0.5], (0.5,1]
    print(f"\n{name} — распределение depd_frac по диапазонам:")
    if df.empty:
        print("  ПУСТОЙ датасет")
        return
    x = df["depd_frac"].values
    mask_0 = (x == 0.0)
    mask_01 = (x > 0.0) & (x <= 0.1)
    mask_05 = (x > 0.1) & (x <= 0.5)
    mask_1 = (x > 0.5) & (x <= 1.0 + 1e-9)

    total = len(x)
    for label, m in [
        ("== 0.0", mask_0),
        ("(0.0, 0.1]", mask_01),
        ("(0.1, 0.5]", mask_05),
        ("(0.5, 1.0]", mask_1),
    ]:
        cnt = m.sum()
        print(f"  {label:10s}: {cnt:6d}  ({cnt/total:6.3%})")

def describe_windows(df, name):
    print(f"\n{name}:")
    if df.empty:
        print("  ПУСТОЙ датасет")
        return
    print("  windows:", len(df))
    print("  depd_frac describe:")
    print(df["depd_frac"].describe())
    print("  >0 count:", (df["depd_frac"] > 0).sum())
    print("  ==0 count:", (df["depd_frac"] == 0).sum())
    print("  >0 share:", (df["depd_frac"] > 0).mean())

In [3]:
rows = []
recording_intervals_norm = {}
continuous_data = {}  # rec_key -> (data_all, sfreq, ch_names)
recording_intervals_fixed = {}
depd_spans_norm = {}
trimmed_continuous_data = {}       # rec_key -> (data_trimmed, sfreq, ch_names, t_lo, t_hi)
trimmed_recording_intervals = {}   # rec_key -> dict с обрезанными sleep/wake/depd
windows_by_record = {}  # rec_key -> {"sleep": df, "wake": df}

for rec_key, rec_root in RECORDINGS.items():
    man = segment_loader.load_export_manifest(rec_root)
    sfreq = float(man.get("sfreq", 0.0))
    recording_id = man.get("recording_id")

    crop_region = man.get("crop_region_global_sec")
    if crop_region and crop_region[0] is not None:
        offset = float(crop_region[0])
    else:
        tmins_tmp = []
        for idx in segment_loader.iter_segment_indices(rec_root):
            meta_tmp = segment_loader.load_segment_meta(rec_root, idx)
            tmins_tmp.append(float(meta_tmp.get("tmin_global_sec", 0.0)))
        offset = min(tmins_tmp) if tmins_tmp else 0.0

    sleep_all = []
    wake_all = []
    depd_all = []

    for idx in segment_loader.iter_segment_indices(rec_root):
        meta = segment_loader.load_segment_meta(rec_root, idx)

        tmin_global = float(meta.get("tmin_global_sec", 0.0))
        tmax_global = float(meta.get("tmax_global_sec", 0.0))

        tmin_norm = tmin_global - offset
        tmax_norm = tmax_global - offset

        sleep_int = meta.get("intervals_sleep", [])
        wake_int = meta.get("intervals_wake", [])
        depd_ch_int = meta.get("intervals_by_channel", [])
        depd_gl_int = meta.get("intervals_global", [])

        # --- сон: локальное время сегмента -> нормированное время записи
        for r in sleep_int:
            s = tmin_norm + float(r["start"])
            e = tmin_norm + float(r["end"])
            if e > s:
                sleep_all.append({"start": s, "end": e})

        # --- бодрствование
        for r in wake_int:
            s = tmin_norm + float(r["start"])
            e = tmin_norm + float(r["end"])
            if e > s:
                wake_all.append({"start": s, "end": e})

        # --- ДЭПД по каналам
        for r in depd_ch_int:
            s = tmin_norm + float(r["start"])
            e = tmin_norm + float(r["end"])
            if e > s:
                depd_all.append({
                    "start": s,
                    "end": e,
                    "channel": int(r["channel"]) if r.get("channel") is not None else None,
                    "seg": idx,
                })

        rows.append({
            "rec_key": rec_key,
            "recording_id": recording_id,
            "subject_id": meta.get("subject_id"),
            "seg": idx,
            "tmin_global": tmin_global,
            "tmax_global": tmax_global,
            "tmin_norm": tmin_norm,
            "tmax_norm": tmax_norm,
            "duration_sec": float(meta.get("duration_sec", 0.0)),
            "sfreq": float(meta.get("sfreq", sfreq)),
            "sleep": has_nonempty(sleep_int),
            "wake": has_nonempty(wake_int),
            "depd_by_ch": has_nonempty(depd_ch_int),
            "depd_global": has_nonempty(depd_gl_int),
            "n_sleep_intervals": len(sleep_int),
            "n_wake_intervals": len(wake_int),
            "n_depd_by_ch": len(depd_ch_int),
            "n_depd_global": len(depd_gl_int),
            "time_offset": offset,
        })

    # объединяем интервалы по записи уже в нормированной оси
    sleep_merged = merge_intervals(sleep_all)
    wake_merged = merge_intervals(wake_all)
    depd_merged = merge_intervals([{"start": x["start"], "end": x["end"]} for x in depd_all])

    recording_intervals_norm[rec_key] = {
        "offset": offset,
        "sleep": sleep_merged,
        "wake": wake_merged,
        "depd": depd_merged,
        "depd_raw": depd_all,   # если захочешь потом смотреть по каналам
    }

df_segments = pd.DataFrame(rows)

for rec_key, rec_root in RECORDINGS.items():
    print(f"\n=== Сборка записи {rec_key} из {rec_root} ===")
    data_all, sfreq, ch_names = build_continuous_recording(rec_key, rec_root)
    continuous_data[rec_key] = (data_all, sfreq, ch_names)
    print(
        f"rec_key={rec_key}, shape={data_all.shape}, "
        f"sfreq={sfreq}, n_channels={data_all.shape[0]}"
    )
    print("Первые каналы:", ch_names[:5])

for rec_key, rec_root in RECORDINGS.items():
    print(f"\n=== Интервалы (FIXED) для записи {rec_key} ===")
    ivals = build_intervals_for_recording_fixed(rec_key, rec_root)
    recording_intervals_fixed[rec_key] = ivals

    def total_duration(intervals):
        return sum(float(r["end"]) - float(r["start"]) for r in intervals)

    sleep_dur = total_duration(ivals["sleep"])
    wake_dur = total_duration(ivals["wake"])
    depd_dur = total_duration(ivals["depd"])

    print(f"  СОН:   {len(ivals['sleep'])} интервалов, суммарно ~{sleep_dur:.1f} с")
    print(f"  БДР:   {len(ivals['wake'])} интервалов, суммарно ~{wake_dur:.1f} с")
    print(f"  ДЭПД:  {len(ivals['depd'])} интервалов, суммарно ~{depd_dur:.1f} с")

for rec_key in RECORDINGS.keys():
    t_lo, t_hi = depd_span_for_recording_norm(rec_key)
    depd_spans_norm[rec_key] = (t_lo, t_hi)
    print(f"{rec_key}: earliest DEPD (norm) = {t_lo}, latest DEPD (norm) = {t_hi}")

for rec_key in RECORDINGS.keys():
    data_all, sfreq, ch_names = continuous_data[rec_key]
    ivals = recording_intervals_norm[rec_key]

    # span по нормированным ДЭПД
    t_lo, t_hi = depd_spans_norm[rec_key]
    if t_lo is None or t_hi is None:
        # нет ДЭПД — можно либо пропустить, либо взять всю запись как есть
        print(f"{rec_key}: нет DEPD в normalized intervals, запись не обрезаем")
        t_lo = 0.0
        t_hi = data_all.shape[1] / sfreq

    # переводим в сэмплы на нормированной оси 0–T
    start_sample = int(round(t_lo * sfreq))
    end_sample = int(round(t_hi * sfreq))
    start_sample = max(0, start_sample)
    end_sample = min(data_all.shape[1], end_sample)

    if end_sample <= start_sample:
        print(f"{rec_key}: WARNING — пустой диапазон после обрезки, пропускаем")
        trimmed_continuous_data[rec_key] = (data_all[:, :0], sfreq, ch_names, 0.0, 0.0)
        trimmed_recording_intervals[rec_key] = {"sleep": [], "wake": [], "depd": [], "span": (0.0, 0.0)}
        continue

    data_trimmed = data_all[:, start_sample:end_sample]
    t_lo_eff = start_sample / sfreq
    t_hi_eff = end_sample / sfreq

    trimmed_continuous_data[rec_key] = (data_trimmed, sfreq, ch_names, t_lo_eff, t_hi_eff)

    sleep_trim = clip_intervals_to_span(ivals["sleep"], t_lo_eff, t_hi_eff)
    wake_trim = clip_intervals_to_span(ivals["wake"], t_lo_eff, t_hi_eff)
    depd_trim = clip_intervals_to_span(ivals["depd"], t_lo_eff, t_hi_eff)

    trimmed_recording_intervals[rec_key] = {
        "sleep": sleep_trim,
        "wake": wake_trim,
        "depd": depd_trim,
        "span": (t_lo_eff, t_hi_eff),
    }

    print(f"\n{rec_key}: trimmed span [{t_lo_eff:.3f}, {t_hi_eff:.3f}] s")
    print(f"  sleep: {len(sleep_trim)} intervals, total ~{total_duration(sleep_trim):.1f} s")
    print(f"  wake:  {len(wake_trim)} intervals, total ~{total_duration(wake_trim):.1f} s")
    print(f"  depd:  {len(depd_trim)} intervals, total ~{total_duration(depd_trim):.1f} s")

for rec_key in RECORDINGS.keys():
    print(f"\n=== Генерация окон для записи {rec_key} ===")

    df_sleep = build_windows_for_recording_subset_trimmed(
        rec_key, subset="sleep", window_sec=0.4, step_sec=0.1
    )
    df_wake = build_windows_for_recording_subset_trimmed(
        rec_key, subset="wake", window_sec=0.4, step_sec=0.1
    )

    windows_by_record[rec_key] = {
        "sleep": df_sleep,
        "wake": df_wake,
    }

    print(f"  {rec_key}_sleep: {len(df_sleep)} окон")
    print(f"  {rec_key}_wake:  {len(df_wake)} окон")

df_000_wake = windows_by_record["000"]["wake"]
df_004_wake = windows_by_record["004"]["wake"]
df_005_sleep = windows_by_record["005"]["sleep"]
df_008_wake = windows_by_record["008"]["wake"]

describe_windows(df_000_wake, "000_wake")
describe_windows(df_004_wake, "004_wake")
describe_windows(df_005_sleep, "005_sleep")
describe_windows(df_008_wake, "008_wake")

frac_bins(df_000_wake, "000_wake")
frac_bins(df_004_wake, "004_wake")
frac_bins(df_005_sleep, "005_sleep")
frac_bins(df_008_wake, "008_wake")



=== Сборка записи 000 из C:\Users\Admin\Desktop\EEG\exports\exports\00000000\FA0183FS ===
rec_key=000, shape=(18, 19249128), sfreq=500.0, n_channels=18
Первые каналы: ['Fp1-F3', 'Fp2-F4', 'F3-C3', 'F4-C4', 'C3-P3']

=== Сборка записи 004 из C:\Users\Admin\Desktop\EEG\exports\exports\004\FA01841G ===
rec_key=004, shape=(18, 1708011), sfreq=500.0, n_channels=18
Первые каналы: ['Fp1-F3', 'Fp2-F4', 'F3-C3', 'F4-C4', 'C3-P3']

=== Сборка записи 005 из C:\Users\Admin\Desktop\EEG\exports\exports\005_22_03_2026\FA0183FS ===
rec_key=005, shape=(18, 1797532), sfreq=500.0, n_channels=18
Первые каналы: ['Fp1-F3', 'Fp2-F4', 'F3-C3', 'F4-C4', 'C3-P3']

=== Сборка записи 008 из C:\Users\Admin\Desktop\EEG\exports\exports\008\GA15011B ===
rec_key=008, shape=(18, 3568515), sfreq=500.0, n_channels=18
Первые каналы: ['Fp1-F3', 'Fp2-F4', 'F3-C3', 'F4-C4', 'C3-P3']

=== Интервалы (FIXED) для записи 000 ===
  СОН:   6 интервалов, суммарно ~13037.8 с
  БДР:   7 интервалов, суммарно ~25460.2 с
  ДЭПД:  192 ин

In [6]:
from typing import List, Tuple, Dict
from scipy.signal import welch, find_peaks
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

# import numpy as np
# import pandas as pd
# from sklearn.metrics import (
#     mean_squared_error, mean_absolute_error, r2_score,
#     accuracy_score, precision_score, recall_score, f1_score,
#     roc_auc_score, average_precision_score, confusion_matrix,
# )


In [7]:
def extract_window_features(window: np.ndarray, fs: float) -> np.ndarray:
    """
    window: (n_channels, n_samples)
    return: 1D feature vector for this window
    """
    n_channels, n_samples = window.shape
    feats = []

    # -------- 1. Морфология по каналам --------
    # базовые статистики
    mean_ch = window.mean(axis=1)
    std_ch = window.std(axis=1)
    max_ch = window.max(axis=1)
    min_ch = window.min(axis=1)
    ptp_ch = max_ch - min_ch  # peak-to-peak

    feats.extend(mean_ch)
    feats.extend(std_ch)
    feats.extend(max_ch)
    feats.extend(min_ch)
    feats.extend(ptp_ch)

    # производная по времени
    diff = np.diff(window, axis=1) * fs  # приблизительная производная
    max_slope_ch = np.max(np.abs(diff), axis=1)
    feats.extend(max_slope_ch)

    # "спайковость": пик / RMS и число выбросов
    rms_ch = np.sqrt((window**2).mean(axis=1) + 1e-12)
    spike_ratio_ch = max_ch / rms_ch
    feats.extend(spike_ratio_ch)

    # выбросы выше k * std относительно медианы
    k = 3.0
    med_ch = np.median(window, axis=1, keepdims=True)
    thr = med_ch + k * std_ch[:, None]
    outliers_count = (window > thr).sum(axis=1)
    feats.extend(outliers_count)

    # -------- 2. Пространственные (дипольные) фичи --------
    # момент максимальной глобальной энергии по окну
    energy_t = np.sum(window**2, axis=0)  # (n_samples,)
    t_peak = np.argmax(energy_t)
    vec_peak = window[:, t_peak]  # (n_channels,)

    max_val = np.max(vec_peak)
    min_val = np.min(vec_peak)
    max_idx = np.argmax(vec_peak)
    min_idx = np.argmin(vec_peak)

    feats.append(max_val)
    feats.append(min_val)
    feats.append(max_val - min_val)
    feats.append(max_idx)
    feats.append(min_idx)

    # нормированное распределение по каналам
    denom = np.max(np.abs(vec_peak)) + 1e-12
    norm_vec_peak = vec_peak / denom
    feats.append(norm_vec_peak.mean())
    feats.append(norm_vec_peak.std())
    feats.append(norm_vec_peak.max())
    feats.append(norm_vec_peak.min())

    # -------- 3. Частотно-энергетические фичи --------
    # PSD по Welch для каждого канала, усредняем по окну
    # (можно подобрать параметры nperseg под твое Fs)
    band_limits = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 70)]
    band_energies = []

    for ch in range(n_channels):
        f, pxx = welch(window[ch], fs=fs, nperseg=min(128, n_samples))
        # pxx: power spectral density
        ch_band_e = []
        for fmin, fmax in band_limits:
            mask = (f >= fmin) & (f < fmax)
            ch_band_e.append(np.trapz(pxx[mask], f[mask]))
        band_energies.append(ch_band_e)

    band_energies = np.array(band_energies)  # (n_channels, n_bands)

    # по каналам: среднее и максимум энергии в каждой полосе
    mean_band = band_energies.mean(axis=0)
    max_band = band_energies.max(axis=0)
    feats.extend(mean_band)
    feats.extend(max_band)

    # отношение "острая/медленная" (13–30 / 1–4)
    slow_idx = 0  # 1–4
    fast_idx = 3  # 13–30
    ratio_fast_slow_ch = (band_energies[:, fast_idx] + 1e-12) / (band_energies[:, slow_idx] + 1e-12)
    feats.extend(ratio_fast_slow_ch)

    return np.asarray(feats, dtype=np.float32)

def build_feature_matrix_for_record_subset(
    rec_key: str,
    subset: str,  # "sleep" или "wake"
    windows_by_record: Dict,
    trimmed_continuous_data: Dict,
    max_windows: int = None,  # опц: ограничить число окон для ускорения
    depd_frac_pos_threshold: float = 0.0,  # >=0.0: берём все, >0: можно оставить только окна с ДЭПД
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Возвращает:
      X_feat: (N, D) — матрица признаков
      y: (N,) — depd_frac
      meta_df: для отладки (индексы, rec_key, subset)
    """
    df = windows_by_record[rec_key][subset]
    if df.empty:
        return None, None, None

    # Фильтр по depd_frac при необходимости
    if depd_frac_pos_threshold > 0.0:
        df = df[df["depd_frac"] >= depd_frac_pos_threshold]
        if df.empty:
            return None, None, None

    data_all, sfreq, ch_names, span_lo, span_hi = trimmed_continuous_data[rec_key]
    n_channels = data_all.shape[0]

    if max_windows is not None and len(df) > max_windows:
        df = df.sample(n=max_windows, random_state=42)

    X_list = []
    y_list = []
    meta_rows = []

    for idx, row in df.iterrows():
        s = int(row["start_sample"])
        e = int(row["end_sample"])
        window = data_all[:, s:e]  # (C, T)
        # safety: пропускаем, если размер не совпал
        if window.shape[1] == 0:
            continue
        feats = extract_window_features(window, fs=sfreq)
        X_list.append(feats)
        y_list.append(float(row["depd_frac"]))
        meta_rows.append({"rec_key": rec_key, "subset": subset, "row_idx": idx})

    if not X_list:
        return None, None, None

    X_feat = np.stack(X_list, axis=0).astype(np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    meta_df = pd.DataFrame(meta_rows)

    print(f"{rec_key}_{subset}: построено {len(y)} окон, размер признаков {X_feat.shape[1]}")
    return X_feat, y, meta_df

def concat_feature_datasets(
    specs: List[Tuple[str, str]],  # [(rec_key, subset), ...]
    windows_by_record: Dict,
    trimmed_continuous_data: Dict,
    max_windows_per_spec: int = None,
    depd_frac_pos_threshold: float = 0.0) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    X_all = []
    y_all = []
    meta_all = []

    for rec_key, subset in specs:
        X_feat, y, meta_df = build_feature_matrix_for_record_subset(
            rec_key, subset, windows_by_record, trimmed_continuous_data,
            max_windows=max_windows_per_spec,
            depd_frac_pos_threshold=depd_frac_pos_threshold,
        )
        if X_feat is None:
            continue
        X_all.append(X_feat)
        y_all.append(y)
        meta_all.append(meta_df)

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    meta = pd.concat(meta_all, axis=0).reset_index(drop=True)
    print(f"Общий датасет: {X.shape[0]} окон, {X.shape[1]} признаков")
    return X, y, meta

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

def eval_regressor_predictions(y_true, y_pred, name, thresholds=(0.2, 0.4, 0.7)):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    print(f"\n=== {name} ===")
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"Regress. MSE={mse:.5f}, MAE={mae:.5f}, R2={r2:.5f}")

    # ROC/PR по задаче y_true>0 (есть ли ДЭПД вообще)
    if (y_true > 0).any():
        try:
            roc = roc_auc_score((y_true > 0).astype(int), y_pred)
            pr = average_precision_score((y_true > 0).astype(int), y_pred)
            print(f"ROC AUC (y_true>0)={roc:.5f}, PR AUC={pr:.5f}")
        except Exception as e:
            print("ROC/PR error:", e)

    for tau in thresholds:
        y_bin_true = (y_true >= tau).astype(int)
        y_bin_pred = (y_pred >= tau).astype(int)
        if y_bin_true.sum() == 0:
            print(f"  tau={tau}: no positive true labels, skip")
            continue
        acc = accuracy_score(y_bin_true, y_bin_pred)
        prec = precision_score(y_bin_true, y_bin_pred, zero_division=0)
        rec = recall_score(y_bin_true, y_bin_pred, zero_division=0)
        f1 = f1_score(y_bin_true, y_bin_pred, zero_division=0)
        print(f"  tau={tau}: Acc={acc:.3f}, Prec={prec:.3f}, Rec={rec:.3f}, F1={f1:.3f}")

def make_sample_weights(y, pos_threshold=0.2, pos_weight=5.0):
    """
    y: depd_frac
    окна с y >= pos_threshold получают вес 1+pos_weight, остальные — 1
    """
    w = np.ones_like(y, dtype=np.float32)
    mask = (y >= pos_threshold)
    w[mask] = 1.0 + pos_weight
    return w

def run_feature_experiment_single(
    rec_key: str,
    subset: str,
    windows_by_record,
    trimmed_continuous_data,
    model_kinds=("rf", "catboost"),
    max_windows: int = None,
    undersample_zero: float = 0.0,  # доля нулей, которую оставим (0.0 — без undersampling)
    weight_pos_threshold: float = 0.2,
    weight_pos_weight: float = 5.0,
):
    print(f"\n\n### Feature experiment for {rec_key}_{subset} ###")

    # сначала берём все окна
    X, y, meta = build_feature_matrix_for_record_subset(
        rec_key, subset,
        windows_by_record, trimmed_continuous_data,
        max_windows=max_windows,
        depd_frac_pos_threshold=0.0,  # здесь берём все, затем undersample по нулям
    )
    if X is None:
        print("Пустой датасет, пропускаем.")
        return

    # optional: undersample нули для ускорения
    if undersample_zero > 0.0:
        mask_zero = (y == 0.0)
        mask_pos = (y > 0.0)
        idx_zero = np.where(mask_zero)[0]
        idx_pos = np.where(mask_pos)[0]

        n_zero_keep = int(len(idx_zero) * undersample_zero)
        if n_zero_keep < len(idx_zero):
            rng = np.random.default_rng(42)
            idx_zero_keep = rng.choice(idx_zero, size=n_zero_keep, replace=False)
            idx_keep = np.concatenate([idx_zero_keep, idx_pos])
            idx_keep.sort()
            X = X[idx_keep]
            y = y[idx_keep]
            meta = meta.iloc[idx_keep].reset_index(drop=True)
            print(f"После undersampling: {len(y)} окон (нулей: {n_zero_keep}, позитивов: {len(idx_pos)})")

    # train/val/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, shuffle=True
    )

    # веса для train
    sample_weight_train = make_sample_weights(
        y_train,
        pos_threshold=weight_pos_threshold,
        pos_weight=weight_pos_weight,
    )

    # Random Forest
    if "rf" in model_kinds:
        print("  -> RandomForestRegressor")
        rf = RandomForestRegressor(
            n_estimators=100,       # можно уменьшить для скорости
            max_depth=12,          # ограничиваем глубину
            max_features="sqrt",   # случайная подвыборка признаков
            n_jobs=-1,
            random_state=42,
        )
        rf.fit(X_train, y_train, sample_weight=sample_weight_train)
        y_pred_rf = rf.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_rf, name=f"{rec_key}_{subset}_RF_feat")

    # CatBoost
    if HAS_CATBOOST and "catboost" in model_kinds:
        print("  -> CatBoostRegressor")
        cb = CatBoostRegressor(
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            iterations=500,
            random_seed=42,
            verbose=False,
        )
        cb.fit(X_train, y_train, sample_weight=sample_weight_train, eval_set=(X_val, y_val))
        y_pred_cb = cb.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_cb, name=f"{rec_key}_{subset}_CatBoost_feat")

def run_feature_experiment_awake_all(
    windows_by_record,
    trimmed_continuous_data,
    model_kinds=("rf", "catboost"),
    max_windows_per_rec: int = None,
    undersample_zero: float = 0.5,
    weight_pos_threshold: float = 0.2,
    weight_pos_weight: float = 5.0,
):
    print("\n\n### Feature experiment awake_all ###")
    specs = [("000","wake"), ("004","wake"), ("008","wake")]

    X, y, meta = concat_feature_datasets(
        specs,
        windows_by_record,
        trimmed_continuous_data,
        max_windows_per_spec=max_windows_per_rec,
        depd_frac_pos_threshold=0.0,
    )
    if X is None:
        print("awake_all пуст.")
        return

    # undersample нули
    if undersample_zero > 0.0:
        mask_zero = (y == 0.0)
        mask_pos = (y > 0.0)
        idx_zero = np.where(mask_zero)[0]
        idx_pos = np.where(mask_pos)[0]
        n_zero_keep = int(len(idx_zero) * undersample_zero)
        if n_zero_keep < len(idx_zero):
            rng = np.random.default_rng(42)
            idx_zero_keep = rng.choice(idx_zero, size=n_zero_keep, replace=False)
            idx_keep = np.concatenate([idx_zero_keep, idx_pos])
            idx_keep.sort()
            X = X[idx_keep]
            y = y[idx_keep]
            meta = meta.iloc[idx_keep].reset_index(drop=True)
            print(f"После undersampling: {len(y)} окон (нулей: {n_zero_keep}, позитивов: {len(idx_pos)})")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, shuffle=True
    )

    sample_weight_train = make_sample_weights(
        y_train,
        pos_threshold=weight_pos_threshold,
        pos_weight=weight_pos_weight,
    )

    if "rf" in model_kinds:
        print("  -> awake_all RF")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        )
        rf.fit(X_train, y_train, sample_weight=sample_weight_train)
        y_pred_rf = rf.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_rf, name="awake_all_RF_feat")

    if HAS_CATBOOST and "catboost" in model_kinds:
        print("  -> awake_all CatBoost")
        cb = CatBoostRegressor(
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            iterations=500,
            random_seed=42,
            verbose=False,
        )
        cb.fit(X_train, y_train, sample_weight=sample_weight_train, eval_set=(X_val, y_val))
        y_pred_cb = cb.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_cb, name="awake_all_CatBoost_feat")

def run_feature_experiment_awake_all(
    windows_by_record,
    trimmed_continuous_data,
    model_kinds=("rf", "catboost"),
    max_windows_per_rec: int = None,
    undersample_zero: float = 0.5,
    weight_pos_threshold: float = 0.2,
    weight_pos_weight: float = 5.0,
):
    print("\n\n### Feature experiment awake_all ###")
    specs = [("000","wake"), ("004","wake"), ("008","wake")]

    X, y, meta = concat_feature_datasets(
        specs,
        windows_by_record,
        trimmed_continuous_data,
        max_windows_per_spec=max_windows_per_rec,
        depd_frac_pos_threshold=0.0,
    )
    if X is None:
        print("awake_all пуст.")
        return

    # undersample нули
    if undersample_zero > 0.0:
        mask_zero = (y == 0.0)
        mask_pos = (y > 0.0)
        idx_zero = np.where(mask_zero)[0]
        idx_pos = np.where(mask_pos)[0]
        n_zero_keep = int(len(idx_zero) * undersample_zero)
        if n_zero_keep < len(idx_zero):
            rng = np.random.default_rng(42)
            idx_zero_keep = rng.choice(idx_zero, size=n_zero_keep, replace=False)
            idx_keep = np.concatenate([idx_zero_keep, idx_pos])
            idx_keep.sort()
            X = X[idx_keep]
            y = y[idx_keep]
            meta = meta.iloc[idx_keep].reset_index(drop=True)
            print(f"После undersampling: {len(y)} окон (нулей: {n_zero_keep}, позитивов: {len(idx_pos)})")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42, shuffle=True
    )

    sample_weight_train = make_sample_weights(
        y_train,
        pos_threshold=weight_pos_threshold,
        pos_weight=weight_pos_weight,
    )

    if "rf" in model_kinds:
        print("  -> awake_all RF")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        )
        rf.fit(X_train, y_train, sample_weight=sample_weight_train)
        y_pred_rf = rf.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_rf, name="awake_all_RF_feat")

    if HAS_CATBOOST and "catboost" in model_kinds:
        print("  -> awake_all CatBoost")
        cb = CatBoostRegressor(
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            iterations=500,
            random_seed=42,
            verbose=False,
        )
        cb.fit(X_train, y_train, sample_weight=sample_weight_train, eval_set=(X_val, y_val))
        y_pred_cb = cb.predict(X_test)
        eval_regressor_predictions(y_test, y_pred_cb, name="awake_all_CatBoost_feat")

In [8]:
def run_awake_all_rf_final(
    windows_by_record,
    trimmed_continuous_data,
    undersample_zero=0.35,
    oversample_pos_factor=2,
    pos_weight=8.0,
):
    print("\n\n### Final RF on awake_all (u, oversample, weights) ###")
    specs = [("000","wake"), ("004","wake"), ("008","wake")]

    X, y, meta = concat_feature_datasets(
        specs,
        windows_by_record,
        trimmed_continuous_data,
        max_windows_per_spec=None,
        depd_frac_pos_threshold=0.0,
    )
    if X is None:
        print("awake_all пуст.")
        return

    print("awake_all:", X.shape, "позитивов доля =", (y > 0).mean())

    rng = np.random.default_rng(42)

    # undersample нулей
    mask_zero = (y == 0.0)
    mask_pos = (y > 0.0)
    idx_zero = np.where(mask_zero)[0]
    idx_pos = np.where(mask_pos)[0]

    n_zero_keep = int(len(idx_zero) * undersample_zero)
    if n_zero_keep < len(idx_zero):
        idx_zero_keep = rng.choice(idx_zero, size=n_zero_keep, replace=False)
    else:
        idx_zero_keep = idx_zero

    idx_keep_base = np.concatenate([idx_zero_keep, idx_pos])

    # oversample позитивов
    if oversample_pos_factor > 0 and len(idx_pos) > 0:
        idx_pos_oversampled = np.repeat(idx_pos, oversample_pos_factor)
        idx_keep = np.concatenate([idx_keep_base, idx_pos_oversampled])
    else:
        idx_keep = idx_keep_base

    rng.shuffle(idx_keep)

    X_bal = X[idx_keep]
    y_bal = y[idx_keep]
    meta_bal = meta.iloc[idx_keep].reset_index(drop=True)

    print("После балансировки:", X_bal.shape, "доля позитивов =", (y_bal > 0).mean())

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_bal, y_bal, test_size=0.2, random_state=42, shuffle=True
    )

    # веса
    w_train = make_sample_weights(
        y_train,
        pos_threshold=0.2,
        pos_weight=pos_weight,
    )

    # модель
    rf = RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
    )
    rf.fit(X_train, y_train, sample_weight=w_train)

    y_test_pred = rf.predict(X_test)

    eval_regressor_predictions(
        y_test,
        y_test_pred,
        name=f"RF_final_awake_all (u={undersample_zero}, o={oversample_pos_factor}, w={pos_weight})",
        thresholds=(0.2, 0.4, 0.7),
    )

    return rf, (X_bal, y_bal, meta_bal)

In [7]:
from transformers import AutoModel
from pathlib import Path
import sys

import numpy as np
import torch

# Позиционный банк
pos_bank = AutoModel.from_pretrained(
    REVE_DIR / "reve-positions",
    trust_remote_code=True
)

# Encoder (начать лучше с base)
reve_model = AutoModel.from_pretrained(
    REVE_DIR / "reve-base",
    trust_remote_code=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reve_model.to(device)
reve_model.eval()

Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

Reve(
  (transformer): TransformerBackbone(
    (layers): ModuleList(
      (0-21): 22 x ModuleList(
        (0): Attention(
          (norm): RMSNorm()
          (to_qkv): Linear(in_features=512, out_features=1536, bias=False)
          (to_out): Linear(in_features=512, out_features=512, bias=False)
          (attend): ClassicalAttention()
        )
        (1): FeedForward(
          (net): Sequential(
            (0): RMSNorm()
            (1): Linear(in_features=512, out_features=2722, bias=False)
            (2): GEGLU()
            (3): Linear(in_features=1361, out_features=512, bias=False)
          )
        )
      )
    )
  )
  (to_patch_embedding): Sequential(
    (0): Linear(in_features=200, out_features=512, bias=True)
  )
  (fourier4d): FourierEmb4D()
  (mlp4d): Sequential(
    (0): Linear(in_features=4, out_features=512, bias=False)
    (1): GELU(approximate='none')
    (2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (ln): LayerNorm((512,), eps=1e-05, el

In [13]:
import torch
import numpy as np

data_000, sfreq_000, ch_names_000, tmin_trim_000, tmax_trim_000 = trimmed_continuous_data["000"]
print("Примеры имен каналов:", ch_names_000[:10])

all_pos_names = set(pos_bank.get_all_positions())
print("Примеры доступных имен в position bank:", list(all_pos_names)[:20])

def normalize_ch_name(raw):
    s = str(raw).strip()
    if "-" in s:
        s = s.split("-")[-1]  # 'Fp1-F3' -> 'F3'
    return s

norm_ch_names_000 = [normalize_ch_name(ch) for ch in ch_names_000]

missing = [ch for ch in norm_ch_names_000 if ch not in all_pos_names]
print("После нормализации не найдены:", missing)

Примеры имен каналов: ['Fp1-F3', 'Fp2-F4', 'F3-C3', 'F4-C4', 'C3-P3', 'C4-P4', 'P3-O1', 'P4-O2', 'Fp1-F7', 'Fp2-F8']
Примеры доступных имен в position bank: ['CPP6h', 'D5', 'O2', 'biosemi128_C15', 'E12', 'biosemi128_C18', 'biosemi128_C24', 'biosemi128_D9', 'E83', 'A7', 'E59', 'AF3', 'AFF6h', 'biosemi128_C7', 'E118', 'biosemi128_B10', 'biosemi128_D20', 'B3', 'biosemi128_D24', 'AFF1h']
После нормализации не найдены: []


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_positions_for_ch_names(ch_names, pos_bank, device):
    norm = [normalize_ch_name(ch) for ch in ch_names]
    all_pos_names = set(pos_bank.get_all_positions())
    missing = [ch for ch in norm if ch not in all_pos_names]
    if missing:
        print("WARNING: не найдены в позиции:", missing)
    pos = pos_bank(norm)  # (C, 3)
    if not isinstance(pos, torch.Tensor):
        pos = torch.as_tensor(pos, dtype=torch.float32)
    return pos.to(device)

positions_000 = get_positions_for_ch_names(ch_names_000, pos_bank, device)
print("positions_000 shape:", positions_000.shape)

positions_000 shape: torch.Size([18, 3])


In [16]:
data_000, sfreq_000, ch_names_000, tmin_trim_000, tmax_trim_000 = trimmed_continuous_data["000"]
df_000_wake = windows_by_record["000"]["wake"]

print("data_000 shape:", data_000.shape)  # (18, N_samples_trim)
print(df_000_wake.shape)
print(df_000_wake.columns)
print(df_000_wake.head())

data_000 shape: (18, 1054282)
(21045, 8)
Index(['rec_key', 'subset', 'start_sec', 'end_sec', 'start_sample',
       'end_sample', 'sfreq', 'depd_frac'],
      dtype='object')
  rec_key subset  start_sec  end_sec  start_sample  end_sample  sfreq  \
0     000   wake      3.676    4.076          1838        2038  500.0   
1     000   wake      3.776    4.176          1888        2088  500.0   
2     000   wake      3.876    4.276          1938        2138  500.0   
3     000   wake      3.976    4.376          1988        2188  500.0   
4     000   wake      4.076    4.476          2038        2238  500.0   

   depd_frac  
0      0.575  
1      0.325  
2      0.075  
3      0.000  
4      0.000  


In [17]:
def windows_to_arrays(df_windows, data_all):
    """
    df_windows: DataFrame с колонками 'start_sample', 'end_sample'
    data_all: np.ndarray (C, T) — trimmed_continuous_data[rec_key][0]
    Возвращает: список np.ndarray (C, window_len)
    """
    arrays = []
    for _, row in df_windows.iterrows():
        s = int(row["start_sample"])
        e = int(row["end_sample"])
        arrays.append(data_all[:, s:e])
    return arrays

windows_000_wake = windows_to_arrays(df_000_wake, data_000)
print(len(windows_000_wake), windows_000_wake[0].shape)

21045 (18, 200)


In [18]:
def normalize_ch_name(raw):
    s = str(raw).strip()
    if "-" in s:
        s = s.split("-")[-1]
    return s

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_positions_for_ch_names(ch_names, pos_bank, device):
    norm = [normalize_ch_name(ch) for ch in ch_names]
    all_pos_names = set(pos_bank.get_all_positions())
    missing = [ch for ch in norm if ch not in all_pos_names]
    if missing:
        print("WARNING: не найдены в позиции:", missing)
    pos = pos_bank(norm)  # (C, 3)
    if not isinstance(pos, torch.Tensor):
        pos = torch.as_tensor(pos, dtype=torch.float32)
    return pos.to(device)

positions_000 = get_positions_for_ch_names(ch_names_000, pos_bank, device)
print("positions_000 shape:", positions_000.shape)  # ожидаем (18, 3)

positions_000 shape: torch.Size([18, 3])


In [25]:
import torch
import numpy as np

@torch.no_grad()
def extract_reve_embeddings_from_arrays(window_arrays, ch_names, reve_model, pos_bank, device, batch_size=32):
    """
    window_arrays: список np.ndarray (C, T)
    ch_names: список каналов (длина C)
    Возвращает:
      X: np.ndarray (N_windows, D)
    """
    positions = get_positions_for_ch_names(ch_names, pos_bank, device)  # (C, 3)

    feats = []
    n = len(window_arrays)
    for start in range(0, n, batch_size):
        batch = window_arrays[start:start+batch_size]
        data = np.stack(batch, axis=0)  # (B, C, T)
        data_t = torch.from_numpy(data).float().to(device)

        pos_batch = positions.unsqueeze(0).expand(data_t.size(0), -1, -1)  # (B, C, 3)

        out = reve_model(data_t, pos_batch)  # (B, C, 1, D) по твоему выводу

        if start == 0:
            print("type(out):", type(out))
            print("out.shape:", out.shape)

        # Приводим к (B, D):
        if out.dim() == 4:
            # (B, C, 1, D) -> (B, C, D)
            out_3d = out.squeeze(2)
            # усредняем по каналам: (B, C, D) -> (B, D)
            pooled = out_3d.mean(dim=1)
        elif out.dim() == 3:
            # на всякий случай, если будущая версия вернёт (B, T, D)
            pooled = out.mean(dim=1)
        elif out.dim() == 2:
            pooled = out
        else:
            raise RuntimeError(f"Неожиданная форма выхода REVE: {out.shape}")

        feats.append(pooled.cpu().numpy())

    return np.concatenate(feats, axis=0)

In [26]:
df_000_sample = df_000_wake.iloc[:200].reset_index(drop=True)
windows_000_sample = windows_to_arrays(df_000_sample, data_000)

X_000_reve = extract_reve_embeddings_from_arrays(
    windows_000_sample,
    ch_names=ch_names_000,
    reve_model=reve_model,
    pos_bank=pos_bank,
    device=device,
    batch_size=16,
)

print("X_000_reve shape:", X_000_reve.shape)

y_000 = df_000_sample["depd_frac"].to_numpy(dtype=np.float32)
print("y_000 shape:", y_000.shape)
print("Примеры y:", y_000[:10])

type(out): <class 'torch.Tensor'>
out.shape: torch.Size([16, 18, 1, 512])
X_000_reve shape: (200, 512)
y_000 shape: (200,)
Примеры y: [0.575 0.325 0.075 0.    0.    0.    0.    0.    0.    0.   ]
